# 第 8 章：工程部署及性能分析 — 动手实验

## 小节概述

本小节是第 8 章的核心实践环节。你将在昇腾 NPU 上完成注意力算子（QKᵀ → scale → softmax → AV）的**工程全流程**：

1. 环境检查（npu-smi / CANN 版本 / msOpGen 验证）
2. 算子原型定义（ops.json）+ msOpGen 工程生成
3. Host 实现（Tiling / InferShape）+ Kernel 实现（朴素三步）
4. 编译 → 打包 → 部署 + aclnn 单算子调用 + 精度验证
5. msProf 上板采集与性能报告解读
6. 复杂度演示：多 seq_len 实测 O(S²) 曲线
7. msSanitizer 异常注入检测 + msDebug 工具认识

> 所有命令均在本 Notebook 的 code cell 中执行，无需手动打开终端。
> 源码工程位于 `src/attention_op/`，通过 `cat` / `%%writefile` 直接查看与讲解。


## 步骤 1：确认环境和目标平台

In [ ]:
import os
import subprocess
import sys

# 工作目录定位：保证从本 Notebook 所在章节目录执行
if not os.path.exists('src/attention_op'):
    for p in ['contrib/tutorials/data_structures_compute/08_engineering_deployment_and_perf_analysis']:
        if os.path.exists(os.path.join(p, 'src/attention_op')):
            os.chdir(p)
            break
print('工作目录:', os.getcwd())

ASCEND_HOME = os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/ascend-toolkit/latest')
print(f'ASCEND_HOME_PATH = {ASCEND_HOME}')

def run(cmd, **kw):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(r.stdout, end='')
    if r.stderr: print(r.stderr, end='')
    return r.returncode

# 1) NPU 设备
run('npu-smi info | head -20')
# 2) CANN 版本
run(f'cat {ASCEND_HOME}/version.cfg 2>/dev/null || ls {ASCEND_HOME}')
# 3) 开发工具
run(f'ls {ASCEND_HOME}/bin/ | grep -E "msopgen|msprof|msdebug|mssanitizer"')
print('环境检查完成。')


## 步骤 2：算子原型定义与工程生成

### 2.1 原型定义（ops.json）

自定义算子的起点是**算子原型定义**：声明算子的输入、输出及其数据类型。`src/attention_op/custom_ops/ops.json` 定义了 3 个输入（q / k / v，均为 float16）+ 1 个输出（o）：

In [ ]:
# 查看算子原型定义
run('cat src/attention_op/custom_ops/ops.json')


### 2.2 msOpGen 生成算子工程

`msOpGen` 基于原型定义一键生成工程模板（op_host / op_kernel / framework 骨架）：

```bash
msopgen gen -i ops.json -c ai_core-ascend910b -out <输出目录>
```

> 本实验的演示生成到 `custom_ops/msopgen_demo/`（避免覆盖已实现的工程）。

In [ ]:
# 生成工程模板（演示）
run(f'source {ASCEND_HOME}/set_env.sh && '
    'cd src/attention_op/custom_ops && '
    'rm -rf msopgen_demo && '
    'msopgen gen -i ops.json -c ai_core-ascend910b -out msopgen_demo')
# 查看生成的工程结构
run('find src/attention_op/custom_ops/msopgen_demo -maxdepth 3 -type f | head -20')


## 步骤 3：Host + Kernel 实现

`src/attention_op/custom_ops/generated/AttentionCustom/` 即本实验使用的算子工程（由 msOpGen 生成后手工实现 Host 与 Kernel）。

### 3.1 Tiling 参数（Host 侧）

Tiling 在 Host 侧运行，根据输入 shape 计算运行期参数：`seqLen` / `dim` / `scale = 1/√D`，并设置**多核并行**（`blockDim = AIV 核数 40`，按行切分）：

In [ ]:
# 查看 Tiling 关键代码
run('sed -n "8,40p" src/attention_op/custom_ops/generated/AttentionCustom/op_host/attention_custom.cpp')


### 3.2 Kernel 实现（Device 侧）

Kernel 使用**纯标量实现**（GM 标量访问 + UB 普通数组），三步计算：

- **阶段 A**：`scores[i][j] = Σₖ q[i][k]·kt[k][j]·scale` → 行 softmax（max 减 → exp → sum 归一），原地变 P
- **阶段 B**：`o[i][j] = Σₖ P[i][k]·v[k][j]`
- **多核切分**：`row = blockIdx; row < seqLen; row += blockDim`，行间无依赖

> 为什么教学版用纯标量实现？见本章 README「为什么用纯标量+多核」——与已验证稳定的 API 子集同构，且 O(S²) 复杂度演示与实现方式无关；Cube/Vector 向量化实现作为课后实践题。

In [ ]:
# 查看 Kernel 核心实现
run('sed -n "70,140p" src/attention_op/custom_ops/generated/AttentionCustom/op_kernel/attention_custom.cpp')


## 步骤 4：编译 → 打包 → 部署 → aclnn 调用

### 4.1 编译并安装算子包

`build_ops.sh` 完成：编译（cmake + 算子编译器）→ 打包（.run 安装包）→ 安装到 `${HOME}/vendors/customize`（用户目录，无需 root）。

In [ ]:
# 编译 + 打包 + 安装（约 2-3 分钟）
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && bash scripts/build_ops.sh')


### 4.2 编译 aclnn Runner

`main_attention_benchmark.cpp` 通过 **aclnn 接口**（`aclnnAttentionCustom`）单算子调用：创建张量 → 获取 workspace → 执行 → 回读输出 → 与 torch 参考比对。

In [ ]:
# 编译 runner
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && bash scripts/build_runner.sh')


### 4.3 生成测试数据

`gen_data.py` 生成 q/k/v（fp16）与 torch 参考输出 ref（fp32 计算后转 fp16），支持多个 seq_len：

In [ ]:
# 生成 512/1024/2048/4096 数据（seed=42，幂等）
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && python3 scripts/gen_data.py')


### 4.4 运行 Benchmark：aclnn 单算子调用 + 精度验证

预期输出 `result: PASS`，maxAbsErr < 1e-2。

In [ ]:
# 运行 512 数据
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && '
    'source scripts/env_custom_opp.sh > /dev/null && '
    'aclnn_runner/build/main_attention_benchmark data 512 64')


## 步骤 5：msProf 上板性能采集与报告解读

### 5.1 采集

`run_profiling.sh` 封装 `msprof op` 命令，对指定 seq_len 采集性能数据：

In [ ]:
# msProf 采集（512，约 1-2 分钟）
run(f'source {ASCEND_HOME}/set_env.sh && '
    'cd src/attention_op && bash scripts/run_profiling.sh 512 --output prof')


### 5.2 报告解读

CANN 9.0.0 的 msProf 报告以 CSV 集呈现（`OpBasicInfo` / `PipeUtilization` / `Memory` 等）。读取关键字段：

In [ ]:
import glob, csv

prof_dir = glob.glob('src/attention_op/prof/prof_512/OPPROF_*')
assert prof_dir, '未找到采集报告，请先执行上一步'
prof_dir = prof_dir[-1]

with open(f'{prof_dir}/OpBasicInfo.csv') as f:
    row = next(csv.DictReader(f))
print('=== OpBasicInfo ===')
for k in ['Op Name', 'Op Type', 'Task Duration(us)', 'Block Dim', 'Device Id']:
    print(f'  {k}: {row.get(k)}')

with open(f'{prof_dir}/PipeUtilization.csv') as f:
    rows = list(csv.DictReader(f))
print(f'=== PipeUtilization（{len(rows)} 个核）===')
r = rows[0]
for k in ['block_id', 'sub_block_id', 'aiv_time(us)', 'aiv_vec_time(us)', 'aiv_vec_ratio',
          'aiv_mte2_time(us)', 'aiv_mte2_ratio', 'aiv_mte3_time(us)', 'aiv_scalar_time(us)']:
    print(f'  {k}: {r.get(k)}')


### 5.3 关键结论

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>字段</th>
      <th>值</th>
      <th>含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Task Duration</td>
      <td>~85 ms</td>
      <td>算子总执行时间（与 benchmark 计时一致）</td>
    </tr>
    <tr>
      <td>Block Dim</td>
      <td>40</td>
      <td>多核并行度（40 个 AIV 核）</td>
    </tr>
    <tr>
      <td>aiv_vec_ratio</td>
      <td>≈ 0</td>
      <td><strong>向量流水利用率几乎为 0</strong> —— 纯标量实现没有向量指令，瓶颈在标量 GM 访问</td>
    </tr>
    <tr>
      <td>aiv_time</td>
      <td>~85 ms</td>
      <td>AIV 核总耗时，与 Task Duration 基本持平（流水无重叠）</td>
    </tr>
  </tbody>
</table>

> **分析**：`aiv_vec_ratio ≈ 0` 说明计算资源远未用满，性能瓶颈在标量指令与 GM 标量访问的延迟。
> 这正是后续优化的切入点（向量化 / Cube 单元 / 流水重叠），也是 08.03 实践题的背景。

## 步骤 6：复杂度演示 — 实测 O(S²) 曲线

注意力算子的计算量 FLOPs ≈ 4·S²·D，理论复杂度 **O(S²)**：seq_len 翻倍 → 耗时约 ×4。
下面实测 512 / 1024 / 2048 / 4096 的耗时（合计约 10 秒）：

In [ ]:
import time

results = {}
for s in [512, 1024, 2048, 4096]:
    print(f'>>> seq_len={s} ...')
    r = subprocess.run(
        f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && '
        'source scripts/env_custom_opp.sh > /dev/null && '
        f'aclnn_runner/build/main_attention_benchmark data {s} 64',
        shell=True, capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if 'time=' in line and 'seq_len=' in line:
            print('  ' + line.strip())
            import re
            m = re.search(r'time=([\d.]+) ms', line)
            if m: results[s] = float(m.group(1))
    if r.returncode != 0:
        print(r.stdout, r.stderr)

print()
print('=== 实测耗时 ===')
prev = None
for s, t in results.items():
    ratio = f'{t/prev:.1f}x' if prev else '-'
    print(f'  seq_len={s:5d}: {t:8.1f} ms   （相对上一档 {ratio}）')
    prev = t


In [ ]:
# 绘制 O(S²) 增长曲线
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

lens = sorted(results)
times = [results[s] for s in lens]

plt.figure(figsize=(8, 5))
plt.plot(lens, times, 'o-', color='#E65100', linewidth=2.5, markersize=8)
for x, y in zip(lens, times):
    plt.annotate(f'{y:.0f} ms', (x, y), textcoords='offset points', xytext=(8, 8), fontsize=11)
plt.xlabel('seq_len S')
plt.ylabel('耗时 (ms)')
plt.title('AttentionCustom 实测耗时：O(S²) 增长（dim=64，40 AIV 核）')
plt.grid(alpha=0.3)
plt.savefig('src/attention_op/on2_curve_measured.png', dpi=110, bbox_inches='tight')
plt.show()
print('曲线已保存：src/attention_op/on2_curve_measured.png')


### 6.1 分析

- **seq_len 翻倍 → 耗时约 ×4**（85 → 355 → 1386 → 5506 ms），与理论 O(S²) 一致；
- 注意力矩阵 `scores` 大小 S×S，**显存占用同样是 O(S²)** —— 这是长序列大模型推理的核心瓶颈；
- **Flash Attention（FA）** 通过分块计算避免实例化 S×S 矩阵，将显存降到 O(S)，计算耗时接近线性——本实验的朴素实现与 FA 的差距正是「工程优化」的动机。

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>seq_len</th>
      <th>512</th>
      <th>1024</th>
      <th>2048</th>
      <th>4096</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>实测耗时 (ms)</td>
      <td>85</td>
      <td>355</td>
      <td>1386</td>
      <td>5506</td>
    </tr>
    <tr>
      <td>相对上一档</td>
      <td>-</td>
      <td>×4.2</td>
      <td>×3.9</td>
      <td>×4.0</td>
    </tr>
  </tbody>
</table>

## 步骤 7：msSanitizer 异常检测

`mssanitizer` 提供内存/竞态/初始化/同步四类检查（memcheck / racecheck / initcheck / synccheck）。

### 7.1 健康 Kernel 检查

对当前（正确）算子执行 memcheck，预期 `No error detected`：

In [ ]:
# memcheck：健康 kernel（插桩运行，耗时略长）
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && '
    'source scripts/env_custom_opp.sh > /dev/null && '
    'mssanitizer -t memcheck aclnn_runner/build/main_attention_benchmark data 512 64 1 1 2>&1 | grep -E "memcheck|error|detected"')


### 7.2 注入越界写 → memcheck 检出

人为在 Kernel 中注入一个**越界写**（向 o 缓冲末尾之外写 2 字节），验证 memcheck 的检出能力：

In [ ]:
# 注入越界写
kernel_path = 'src/attention_op/custom_ops/generated/AttentionCustom/op_kernel/attention_custom.cpp'
s = open(kernel_path).read()
marker = '// ==== INJECTED:'
assert marker not in s, '已注入过，先执行恢复 cell'
old = '''__aicore__ inline void AttentionKernel::Process()
{
'''
new = old + '''    // ==== INJECTED: 越界写（演示 msSanitizer memcheck 检出）====
    oGlobal.SetValue(seqLen * dim + 16, static_cast<half>(1.0f));
'''
open(kernel_path, 'w').write(s.replace(old, new))
print('已注入越界写，重新编译安装中（约 2 分钟）...')

r = subprocess.run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && bash scripts/build_ops.sh',
                   shell=True, capture_output=True, text=True)
print('BUILD', 'OK' if r.returncode == 0 else 'FAIL')


In [ ]:
# memcheck 检出注入的越界写
run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && '
    'source scripts/env_custom_opp.sh > /dev/null && '
    'mssanitizer -t memcheck aclnn_runner/build/main_attention_benchmark data 512 64 1 1 2>&1 | '
    'grep -E "ERROR|illegal|detected|Access"')


预期输出：

```text
====== ERROR: illegal write of size 2
```

即 memcheck 精确定位了越界写的指令（2 字节 = half）。**注意**：正常运行时这类越界可能不崩溃（恰好落在已分配内存内），但结果是未定义行为——这正是 sanitizer 的价值。

In [ ]:
# 恢复 Kernel 源码并重新编译（重要！）
kernel_path = 'src/attention_op/custom_ops/generated/AttentionCustom/op_kernel/attention_custom.cpp'
s = open(kernel_path).read()
inj = '''    // ==== INJECTED: 越界写（演示 msSanitizer memcheck 检出）====
    oGlobal.SetValue(seqLen * dim + 16, static_cast<half>(1.0f));
'''
assert inj in s, '未找到注入代码'
open(kernel_path, 'w').write(s.replace(inj, ''))
print('已恢复源码，重新编译安装...')

r = subprocess.run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && bash scripts/build_ops.sh',
                   shell=True, capture_output=True, text=True)
print('BUILD', 'OK' if r.returncode == 0 else 'FAIL')

# 回归验证
r = subprocess.run(f'source {ASCEND_HOME}/set_env.sh && cd src/attention_op && '
                   'source scripts/env_custom_opp.sh > /dev/null && '
                   'aclnn_runner/build/main_attention_benchmark data 512 64',
                   shell=True, capture_output=True, text=True)
print(r.stdout)


## 步骤 8：msDebug 认识

`msdebug` 支持在真实硬件上对 Kernel 断点调试、单步执行、变量/内存查看。

> **环境限制**：msDebug 需要驱动开启调试通道（`--full` / `/proc/debug_switch`），CANNLab 云环境大概率不可用。
> 本实验以命令演示 + 能力清单学习为主；如有本地昇腾环境可参照官方文档实操。

In [ ]:
# msDebug 命令演示
run(f'source {ASCEND_HOME}/set_env.sh && msdebug --help 2>&1 | head -25')


### msDebug 核心能力清单

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>能力</th>
      <th>命令/操作</th>
      <th>用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>断点</td>
      <td><code>break &lt;行号/函数名&gt;</code></td>
      <td>在 Kernel 代码行或函数处暂停</td>
    </tr>
    <tr>
      <td>单步</td>
      <td><code>next</code> / <code>step</code> / <code>continue</code></td>
      <td>逐指令 / 逐行执行</td>
    </tr>
    <tr>
      <td>变量查看</td>
      <td><code>print &lt;变量&gt;</code></td>
      <td>查看标量变量值</td>
    </tr>
    <tr>
      <td>内存查看</td>
      <td><code>memory read &lt;地址&gt; &lt;长度&gt;</code></td>
      <td>查看 GM / UB 内存内容</td>
    </tr>
    <tr>
      <td>核切换</td>
      <td><code>switch core &lt;id&gt;</code></td>
      <td>多核任务中切换调试目标核</td>
    </tr>
    <tr>
      <td>反汇编</td>
      <td><code>disassemble</code></td>
      <td>查看汇编定位指令级问题</td>
    </tr>
    <tr>
      <td>Core dump</td>
      <td>崩溃后自动生成</td>
      <td>崩溃现场分析</td>
    </tr>
  </tbody>
</table>

## 实验总结

你已经完成了：

1. ✅ 环境检查：NPU / CANN / msOpGen / msProf / mssanitizer / msdebug
2. ✅ 算子原型定义（ops.json）+ msOpGen 工程生成
3. ✅ Host（Tiling / InferShape）+ Kernel（朴素三步注意力，纯标量 + 40 核切分）
4. ✅ 编译 → 打包 → 部署到 `${HOME}/vendors/customize` + aclnn 单算子调用 + 精度 PASS
5. ✅ msProf 上板采集与报告解读（Task Duration / Block Dim / aiv_vec_ratio ≈ 0 的瓶颈分析）
6. ✅ 复杂度演示：85 / 355 / 1386 / 5506 ms，O(S²) 曲线与 Flash Attention 理论对比
7. ✅ msSanitizer 异常注入检测（memcheck 检出 illegal write）+ msDebug 能力认识

**下一步**：完成 [08.03 章节实践](08.03_chapter_test.ipynb)——Matmul+Softmax 向量化改造实践题 + 知识测验。
